In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os
import copy
from tqdm import tqdm # For Keras-like progress bars

# 1. Setup Device (Will automatically target your RTX 4060)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Define Directories and Hyperparameters
base_dir = 'dataset/Data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15

# 3. Data Augmentation and Preprocessing
# PyTorch requires manual normalization using ImageNet statistics
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(12), # Roughly 0.2 radians
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 4. Load Datasets
print("Loading Datasets...")
image_datasets = {
    'train': datasets.ImageFolder(train_dir, data_transforms['train']),
    'val': datasets.ImageFolder(val_dir, data_transforms['val']),
    'test': datasets.ImageFolder(test_dir, data_transforms['test'])
}

# Use pin_memory=True to speed up data transfer to the GPU
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True),
    'val': DataLoader(image_datasets['val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True),
    'test': DataLoader(image_datasets['test'], batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
}

class_names = image_datasets['train'].classes
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
print(f"Classes: {len(class_names)}")

# 5. Build the Model (EfficientNetV2-S for SOTA accuracy)
print("Downloading and configuring EfficientNetV2-S...")
weights = models.EfficientNet_V2_S_Weights.DEFAULT
model = models.efficientnet_v2_s(weights=weights)

# Freeze the base model (Optional: unfreeze for even higher accuracy, but training will be slower)
for param in model.parameters():
    param.requires_grad = False

# Reconfigure the classification head for 38 classes
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(num_ftrs, len(class_names))
)

model = model.to(device)

# 6. Loss, Optimizer, and Learning Rate Scheduler
criterion = nn.CrossEntropyLoss()
# AdamW generally yields better generalization than standard Adam
optimizer = optim.AdamW(model.classifier.parameters(), lr=0.001) 
# Reduce learning rate if validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

# 7. Training Loop with Early Stopping
best_model_wts = copy.deepcopy(model.state_dict())
best_loss = float('inf')
patience = 3
patience_counter = 0

print("Starting training...")
for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        # Progress bar
        dataloader_pbar = tqdm(dataloaders[phase], desc=phase.capitalize(), leave=False)

        for inputs, labels in dataloader_pbar:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
            # Update progress bar
            dataloader_pbar.set_postfix({'loss': loss.item()})

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        if phase == 'val':
            scheduler.step(epoch_loss)
            
            # Early Stopping Logic
            if epoch_loss < best_loss:
                best_loss = epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                
    if patience_counter >= patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs.")
        break

# Load best model weights
model.load_state_dict(best_model_wts)

# 8. Evaluation on Test Data
print("\nEvaluating on Test Data...")
model.eval()
running_corrects = 0

with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc="Testing"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == labels.data)

test_acc = running_corrects.double() / dataset_sizes['test']
print(f'Test Accuracy: {test_acc*100:.2f}%')

# 9. Save the Model
save_path = 'plant_disease_model.pth'
torch.save(model.state_dict(), save_path)
print(f"Model successfully saved as '{save_path}'!")

Using device: cuda
Loading Datasets...
Classes: 38


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to C:\Users\bened/.cache\torch\hub\checkpoints\efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:35<00:00, 2.41MB/s]


Starting training...

Epoch 1/15
----------


Train Loss: 1.1025 Acc: 0.7265


Val Loss: 0.5552 Acc: 0.8490

Epoch 2/15
----------


...

Val Loss: 0.3340 Acc: 0.8903

Epoch 15/15
----------


Train Loss: 0.5354 Acc: 0.8345


Val Loss: 0.3299 Acc: 0.8935

Evaluating on Test Data...


Testing: 100%|██████████| 171/171 [00:20<00:00,  8.29it/s]

Test Accuracy: 89.54%
Model successfully saved as 'plant_disease_model.pth'!


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os
import copy
from tqdm import tqdm

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Directories and Hyperparameters
base_dir = 'dataset/Data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25 # Increased max epochs since we are fine-tuning

# 3. Enhanced Data Augmentation
# Added Vertical Flip and Color Jitter to simulate outdoor lighting/shadows
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(), # Plants can be photographed from any angle
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 4. Load Datasets
print("Loading Datasets...")
image_datasets = {x: datasets.ImageFolder(os.path.join(base_dir, x), data_transforms[x]) 
                  for x in ['train', 'val', 'test']}

dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, 
                             shuffle=(x == 'train'), num_workers=4, pin_memory=True) 
               for x in ['train', 'val', 'test']}

class_names = image_datasets['train'].classes
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}

# 5. Build Model: Unfrozen EfficientNetV2-S
print("Configuring EfficientNetV2-S for Full Fine-Tuning...")
weights = models.EfficientNet_V2_S_Weights.DEFAULT
model = models.efficientnet_v2_s(weights=weights)

# UNFREEZE the base model - This is the key to 95%+ accuracy
for param in model.parameters():
    param.requires_grad = True

# Reconfigure the classification head
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True), # Slightly higher dropout since all layers are training
    nn.Linear(num_ftrs, len(class_names))
)

model = model.to(device)

# 6. Loss, Optimizer, Scheduler, and Mixed Precision Scaler
criterion = nn.CrossEntropyLoss()

# VERY IMPORTANT: Use a smaller learning rate (1e-4) because the model is unfrozen.
# We don't want large gradient steps to wreck the pre-trained ImageNet weights.
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4) 

# More generous patience to let the deep layers adapt
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# Scaler for RTX 4000 series hardware acceleration
scaler = torch.cuda.amp.GradScaler()

# 7. Training Loop with AMP (Automatic Mixed Precision)
best_model_wts = copy.deepcopy(model.state_dict())
best_loss = float('inf')
patience = 5 # Increased patience
patience_counter = 0

print("Starting training...")
for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        dataloader_pbar = tqdm(dataloaders[phase], desc=phase.capitalize(), leave=False)

        for inputs, labels in dataloader_pbar:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            # Enable Autocast for Mixed Precision on RTX 4060
            with torch.set_grad_enabled(phase == 'train'):
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                if phase == 'train':
                    # Scale gradients and step
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            dataloader_pbar.set_postfix({'loss': loss.item()})

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

        if phase == 'val':
            scheduler.step(epoch_loss)
            
            if epoch_loss < best_loss:
                best_loss = epoch_loss
                best_model_wts = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                
    if patience_counter >= patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs.")
        break

# Load best weights
model.load_state_dict(best_model_wts)

# 8. Evaluation
print("\nEvaluating on Test Data...")
model.eval()
running_corrects = 0

with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc="Testing"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Inference also benefits from AMP
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
        running_corrects += torch.sum(preds == labels.data)

test_acc = running_corrects.double() / dataset_sizes['test']
print(f'Test Accuracy: {test_acc*100:.2f}%')

# 9. Save SOTA Model
save_path = 'plant_disease_model_sota.pth'
torch.save(model.state_dict(), save_path)
print(f"Model successfully saved as '{save_path}'!")

Using device: cuda
Loading Datasets...
Configuring EfficientNetV2-S for Full Fine-Tuning...


C:\Users\bened\AppData\Local\Temp\ipykernel_15244\973851565.py:89: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Starting training...

Epoch 1/25
----------


Train Loss: 0.3721 Acc: 0.9089


Val Loss: 0.0434 Acc: 0.9867

Epoch 2/25
----------


...

Val Loss: 0.0087 Acc: 0.9969

Epoch 15/25
----------


Train Loss: 0.0027 Acc: 0.9992


Val Loss: 0.0101 Acc: 0.9972

Early stopping triggered after 15 epochs.

Evaluating on Test Data...


Testing: 100%|██████████| 171/171 [00:16<00:00, 10.33it/s]

Test Accuracy: 99.63%
Model successfully saved as 'plant_disease_model_sota.pth'!
